In [1]:
from Bio import SeqIO
import gzip
import re
import pandas as pd

In [2]:
# -------- Parameters --------

motif = "CGAACNNNNGTTCG"

max_mismatches = 1

upstream_start = -400
upstream_end = -10

# Maximum intergenic distance for operon prediction
max_gap = 100

# -------- Input files ---------
genome_file = "AM1001-oriented.fasta"
gff_file = "annot.gff"
output_file = "motif_hits_operons_corrected.csv"


In [3]:
# ============================================================
# 2. FUNÇÕES AUXILIARES
# ============================================================

def hamming_distance(seq1, seq2):
    """Calcula o número de diferenças entre duas sequências."""
    return sum(a != b for a, b in zip(seq1, seq2))


def motif_match_with_N(sequence, motif):
    """
    Compara uma sequência com o motivo.
    A letra N no motivo funciona como curinga.
    Retorna o número de mismatches.
    """

    mismatches = 0

    for base, expected in zip(sequence, motif):

        if expected != "N" and base != expected:
            mismatches += 1

    return mismatches


def reverse_complement(sequence):
    """Retorna a sequência complementar reversa."""
    complement = str.maketrans("ATCGatcg", "TAGCtagc")
    return sequence.translate(complement)[::-1]


def load_genome(genome_file):
    """Carrega as sequências do genoma em um dicionário."""

    genome = {}

    for record in SeqIO.parse(genome_file, "fasta"):
        genome[record.id] = str(record.seq).upper()

    return genome


# ============================================================
# 3. IDENTIFICAÇÃO DE OPERONS PUTATIVOS
# ============================================================



def parse_gtf_tss(gff_file, max_gap=100):
    """
    Identifica grupos de genes potencialmente pertencentes
    ao mesmo operon.

    Retorna:
    chrom, tss, strand, operon_id, gene_ids
    """

    genes = []

    opener = gzip.open if gff_file.endswith(".gz") else open
    mode = "rt" if gff_file.endswith(".gz") else "r"

    with opener(gff_file, mode) as handle:

        for line in handle:

            if line.startswith("#"):
                continue

            fields = line.strip().split("\t")

            if len(fields) != 9:
                continue

            chrom = fields[0]
            feature = fields[2]

            if feature != "gene":
                continue

            start = int(fields[3])
            end = int(fields[4])
            strand = fields[6]
            attributes = fields[8]

            if strand not in ["+", "-"]:
                continue

            # Organizar atributos do GFF
            attr_dict = {}

            for item in attributes.split(";"):

                if "=" in item:
                    key, value = item.split("=", 1)
                    attr_dict[key] = value

            # Priorizar nome funcional
            gene_id = (
                attr_dict.get("gene")
                or attr_dict.get("Name")
                or attr_dict.get("locus_tag")
                or attr_dict.get("ID")
            )

            if gene_id is None:
                gene_id = f"{chrom}_{start}_{end}"

            genes.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "strand": strand,
                "gene_id": gene_id
            })

    # Ordenar genes por posição genômica
    genes.sort(key=lambda x: (x["chrom"], x["start"]))

    operons = []
    current_group = []

    for gene in genes:

        if not current_group:
            current_group.append(gene)
            continue

        previous = current_group[-1]

        gap = gene["start"] - previous["end"] - 1

        same_contig = gene["chrom"] == previous["chrom"]
        same_strand = gene["strand"] == previous["strand"]

        if same_contig and same_strand and gap <= max_gap:
            current_group.append(gene)

        else:
            operons.append(current_group)
            current_group = [gene]

    if current_group:
        operons.append(current_group)

    # Criar identificadores e estimar TSS
    results = []

    for i, group in enumerate(operons, start=1):

        strand = group[0]["strand"]
        chrom = group[0]["chrom"]

        if strand == "+":
            tss = min(gene["start"] for gene in group)
            gene_ids = [gene["gene_id"] for gene in group]

        else:
            tss = max(gene["end"] for gene in group)

            gene_ids = [
                gene["gene_id"]
                for gene in sorted(
                    group,
                    key=lambda x: x["start"],
                    reverse=True
                )
            ]

        operon_id = f"OP{i:04d}"

        results.append((
            chrom,
            tss,
            strand,
            operon_id,
            gene_ids
        ))

    return results


# ============================================================
# 4. BUSCA DE MOTIVOS PRÓXIMOS AOS TSS
# ============================================================

def find_motifs_near_tss(
    genome,
    operons,
    motif,
    max_mismatches=1,
    upstream_start=-400,
    upstream_end=10
):

    results = []
    motif_length = len(motif)

    for chrom, tss, strand, operon_id, gene_ids in operons:

        if chrom not in genome:
            continue

        sequence = genome[chrom]
        sequence_length = len(sequence)

        # ----------------------------------------------------
        # Extrair região genômica correspondente à janela
        # ----------------------------------------------------

        if strand == "+":

            # Conversão de coordenadas GFF para Python
            region_start = max(0, tss + upstream_start - 1)
            region_end = min(
                sequence_length,
                tss + upstream_end
            )

            region = sequence[region_start:region_end]

        else:

            region_start = max(
                0,
                tss - upstream_end - 1
            )

            region_end = min(
                sequence_length,
                tss - upstream_start
            )

            region = sequence[region_start:region_end]

            # Colocar a região na orientação da transcrição
            region = reverse_complement(region)

        # Não há sequência suficiente para encontrar o motivo
        if len(region) < motif_length:
            continue

        # ----------------------------------------------------
        # Procurar o motivo
        # ----------------------------------------------------

        for i in range(len(region) - motif_length + 1):

            candidate = region[i:i + motif_length]

            mismatches = motif_match_with_N(candidate, motif)

            if mismatches > max_mismatches:
                continue

            # ------------------------------------------------
            # Calcular posição relativa ao TSS e coordenadas
            # genômicas do motivo
            # ------------------------------------------------

            if strand == "+":

                motif_start = region_start + i + 1
                motif_end = motif_start + motif_length - 1

                relative_position = motif_start - tss

            else:

                motif_end = region_end - i
                motif_start = motif_end - motif_length + 1

                relative_position = tss - motif_end

            # ------------------------------------------------
            # Garantir que o motivo inteiro esteja na janela
            # ------------------------------------------------

            relative_end = relative_position + motif_length - 1

            if relative_position < upstream_start:
                continue

            if relative_end > upstream_end:
                continue

            results.append({

                "operon_id": operon_id,
                "genes_in_operon": ",".join(gene_ids),
                "n_genes": len(gene_ids),

                "chrom": chrom,
                "strand": strand,
                "tss": tss,

                "motif_start": motif_start,
                "motif_end": motif_end,

                "rel_position_motif": relative_position,

                "mismatches": mismatches,
                "matched_seq": candidate

            })

    return pd.DataFrame(results)


# ============================================================
# 5. EXECUÇÃO PRINCIPAL
# ============================================================

def main():

    print("Carregando genoma...")

    genome = load_genome(genome_file)

    print(f"Contigs carregados: {len(genome)}")

    print("\nIdentificando operons putativos...")

    operons = parse_gtf_tss(
        gff_file,
        max_gap=max_gap
    )

    print(f"Operons identificados: {len(operons)}")

    print("\nProcurando motivos próximos aos TSS...")

    df = find_motifs_near_tss(
        genome=genome,
        operons=operons,
        motif=motif,
        max_mismatches=max_mismatches,
        upstream_start=upstream_start,
        upstream_end=upstream_end
    )

    # --------------------------------------------------------
    # Ordenar resultados
    # --------------------------------------------------------

    if not df.empty:

        df = df.sort_values(
            by=["chrom", "tss", "rel_position_motif"]
        )

        df.to_csv(
            output_file,
            index=False
        )

        print(f"\nMotivos encontrados: {len(df)}")

        print("\nDistribuição de mismatches:")
        print(df["mismatches"].value_counts().sort_index())

        print("\nMotivos por operon:")
        print(
            df[
                [
                    "operon_id",
                    "genes_in_operon",
                    "strand",
                    "tss",
                    "rel_position_motif",
                    "mismatches",
                    "matched_seq"
                ]
            ].to_string(index=False)
        )

        print(f"\nResultados salvos em: {output_file}")

    else:

        print("\nNenhum motivo encontrado.")

    # --------------------------------------------------------
    # Verificar se todos os resultados estão dentro da janela
    # --------------------------------------------------------

    if not df.empty:

        invalid = df[
            (df["rel_position_motif"] < upstream_start) |
            (
                df["rel_position_motif"] +
                len(motif) - 1 > upstream_end
            )
        ]

        print("\nVerificação da janela:")

        if invalid.empty:
            print("Todos os motivos estão dentro da janela definida.")

        else:
            print("Atenção: foram encontrados motivos fora da janela.")
            print(invalid)

In [4]:
# ============================================================
# 6. INICIAR ANÁLISE
# ============================================================

if __name__ == "__main__":
    main()

Carregando genoma...
Contigs carregados: 1

Identificando operons putativos...
Operons identificados: 1970

Procurando motivos próximos aos TSS...

Motivos encontrados: 17

Distribuição de mismatches:
mismatches
0     1
1    16
Name: count, dtype: int64

Motivos por operon:
operon_id                    genes_in_operon strand     tss  rel_position_motif  mismatches    matched_seq
   OP0057                     pgaptmp_000183      -  156772                -395           1 CGAACGCGCGTTTG
   OP0139      pgaptmp_000328,pgaptmp_000329      +  302003                -309           1 CGTACCACAGTTCG
   OP0157      pgaptmp_000358,pgaptmp_000359      +  334824                -309           1 CGTACCACAGTTCG
   OP0306      pgaptmp_000620,pgaptmp_000621      +  589055                -259           1 CGCACTGGAGTTCG
   OP0424      pgaptmp_000836,pgaptmp_000837      +  823566                -309           1 CGAACCGGAATTCG
   OP0664                     pgaptmp_001262      - 1210101                 -29    

In [34]:

import gzip

gff_file = "annot.gff"

# Identificadores que apareceram nos resultados
target_ids = {
    "gene-pgaptmp_001712",
    "gene-pgaptmp_001803",
    "gene-pgaptmp_001802",
    "gene-pgaptmp_001804",
    "gene-pgaptmp_001805",
    "gene-pgaptmp_001806",
    "gene-pgaptmp_003373",
    "gene-pgaptmp_003372"
}

opener = gzip.open if gff_file.endswith(".gz") else open
mode = "rt" if gff_file.endswith(".gz") else "r"

with opener(gff_file, mode) as handle:

    for line in handle:

        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        if len(fields) != 9:
            continue

        feature = fields[2]
        attributes = fields[8]

        # Transformar atributos em dicionário
        attr_dict = {}

        for item in attributes.split(";"):
            if "=" in item:
                key, value = item.split("=", 1)
                attr_dict[key] = value

        feature_id = attr_dict.get("ID")
        parent = attr_dict.get("Parent")

        if feature_id in target_ids or parent in target_ids:

            print(
                f"\nFeature: {feature}",
                f"\nID: {feature_id}",
                f"\nParent: {parent}",
                f"\nGene: {attr_dict.get('gene')}",
                f"\nName: {attr_dict.get('Name')}",
                f"\nProduct: {attr_dict.get('product')}",
                f"\nLocus tag: {attr_dict.get('locus_tag')}"
            )



Feature: gene 
ID: gene-pgaptmp_001712 
Parent: None 
Gene: recA 
Name: recA 
Product: None 
Locus tag: pgaptmp_001712

Feature: CDS 
ID: cds-pgaptmp_001712 
Parent: gene-pgaptmp_001712 
Gene: recA 
Name: extdb:pgaptmp_001712 
Product: recombinase RecA 
Locus tag: pgaptmp_001712

Feature: gene 
ID: gene-pgaptmp_001802 
Parent: None 
Gene: None 
Name: pgaptmp_001802 
Product: None 
Locus tag: pgaptmp_001802

Feature: CDS 
ID: cds-pgaptmp_001802 
Parent: gene-pgaptmp_001802 
Gene: None 
Name: extdb:pgaptmp_001802 
Product: hypothetical protein 
Locus tag: pgaptmp_001802

Feature: gene 
ID: gene-pgaptmp_001803 
Parent: None 
Gene: lexA 
Name: lexA 
Product: None 
Locus tag: pgaptmp_001803

Feature: CDS 
ID: cds-pgaptmp_001803 
Parent: gene-pgaptmp_001803 
Gene: lexA 
Name: extdb:pgaptmp_001803 
Product: transcriptional repressor LexA 
Locus tag: pgaptmp_001803

Feature: gene 
ID: gene-pgaptmp_001804 
Parent: None 
Gene: yneA 
Name: yneA 
Product: None 
Locus tag: pgaptmp_001804

Feature: